# Powder data reduction example 

This notebook demonstrates a prototype of powder data reduction workflow for POWTEX using data measured for one mounting unit of the POWTEX detector ([Houben, A., Meinerzhagen, Y., Nachtigall, N., Jacobs, P. & Dron­skowski, R. (2023). J. Appl. Cryst. 56, 633–642](https://doi.org/10.1107/S1600576723002819)) at POWGEN@SNS ([A. Huq, M. J. Kirkham, P. F. Peterson, J. P. Hodges,
P. Whitfield, K. Page, T. Huegle, E. B. Iverson, A. Parizzi & G. Rennich, J. Appl. Crystallogr., 2019, 52, 1189–1201](https://doi.org/10.1107/S160057671901121X)). 

## Imports

In [ ]:
from mlz.diffraction.powder.workflow import PowderReduceP2D
from mlz.diffraction.powder.types import *
from sciline.reporter import RichReporter

## Configure workflow

The workflow diagram below summarises the main reduction steps, ordered from left to right. Red boxes highlight input parameters that are still missing and must be specified before the workflow can run. Some input parameters (e.g. DperpBins, DspacingBins) have default values that can be overridden.

In [ ]:
powder_p2d_workflow = PowderReduceP2D()
powder_p2d_workflow.visualize(graph_attr={"rankdir": "LR"})

In [ ]:
# TODO: files to be uploaded to the Scipp data server and loaded from there instead of local path
file_path = "/home/koshchii/Instruments/POWTEX/Scipp/data/yannick/converted"
# set filenames for loading data
sample_filename = "PG3_48441.nxs_scipp.h5"
empty_filename = "PG3_48398.nxs_scipp.h5"
vanadium_filename = "PG3_48431.nxs_scipp.h5"
empty_vanadium_filename = None
calibration_filename = "PG3_PAC_d48441_2021_2_18_Combined_scipp_calibration.h5"

# set filepaths for loading data
powder_p2d_workflow[Filepath[SampleRun]] = file_path
powder_p2d_workflow[Filepath[EmptyCanRun]] = file_path
powder_p2d_workflow[Filepath[VanadiumRun]] = file_path
powder_p2d_workflow[Filepath[EmptyVanadiumCanRun]] = file_path
powder_p2d_workflow[Filepath[CalibrationRun]] = file_path

# set filenames for loading data
powder_p2d_workflow[Filename[SampleRun]] = sample_filename
powder_p2d_workflow[Filename[EmptyCanRun]] = empty_filename
powder_p2d_workflow[Filename[VanadiumRun]] = vanadium_filename
powder_p2d_workflow[Filename[EmptyVanadiumCanRun]] = empty_vanadium_filename
powder_p2d_workflow[Filename[CalibrationRun]] = calibration_filename

# set full filename (including path) for p2d output
powder_p2d_workflow[P2DOutputFilename] = "./reduced.p2d"

Below, we confirm that setting the filenames & filepaths removes the corresponding red boxes in the workflow diagram

In [ ]:
powder_p2d_workflow.visualize(graph_attr={"rankdir": "LR"})

## Setting d-spacing bins

Here, we override default uniform binning data for dspacing using a dedicated implementation of logarithmic binning and provide a custom uniform binning for dperp

In [ ]:
from mlz.diffraction.powder.data import sc_geomarange

powder_p2d_workflow[DspacingBins] = sc_geomarange("dspacing", 0.4, 2.0, 0.0005, endpoint=True, unit="angstrom")
powder_p2d_workflow[DperpBins] = sc.linspace("dperp", 0.2, 4.0, 191, unit="angstrom")


### Raw sample data in  (d, dperp) space

Let us now take a look at the raw sample data in a 2D (d, dperp) plane

In [ ]:
reporter = RichReporter()
dspacing_2d_sample = powder_p2d_workflow.compute(tp=Dspacing2D[SampleRun], reporter=reporter)
dspacing_2d_sample.plot(norm='log')

## Setting uncertainty broadcast

UncertaintyBroadcastMode needs to be dropped, because otherwise the correlations would be incorrectly propagated. When UncertaintyBroadcastMode.drop is selected, the variances are dropped. See https://journals.sagepub.com/doi/10.3233/JNR-220049 for details. The "upper bounds" mode is not yet implemented.

In [ ]:
powder_p2d_workflow[UncertaintyBroadcastMode] = UncertaintyBroadcastMode.drop
powder_p2d_workflow.visualize(graph_attr={"rankdir": "LR"})

## Setting masks

To properly reduce this dataset, we need to mask some of the data. Below, we load predefined masks and assign them to MaskingData. These masks will be combined and attached to the resulting intensity dataset

In [ ]:
from mlz.diffraction.powder.data import (
    load_powgen_tof_mask,
    load_powgen_wavelength_mask,
    load_powgen_two_theta_mask,
    load_powgen_dspacing_mask,
    load_powgen_dperp_mask
)

# setting various types of masks to exclude certain data points from the final output
powder_p2d_workflow[MaskingData[TofMask]] = load_powgen_tof_mask()
powder_p2d_workflow[MaskingData[WavelengthMask]] = load_powgen_wavelength_mask()
powder_p2d_workflow[MaskingData[TwoThetaMask]] = load_powgen_two_theta_mask()
powder_p2d_workflow[MaskingData[DSpacingMask]] = load_powgen_dspacing_mask()
powder_p2d_workflow[MaskingData[DperpMask]] = load_powgen_dperp_mask()

In [ ]:
powder_p2d_workflow.visualize(graph_attr={"rankdir": "LR"})

## Results

Below, we visualise the reduced 2d intensity data in a (d, dperp) plane 

In [ ]:
reporter = RichReporter()
intensity_2d = powder_p2d_workflow.compute(Intensity2D, reporter=reporter)
intensity_2d.plot(norm='log')

Corresponding intensity in a p2d format (with masks applied) can then be saved to a scipp.DataArray

In [ ]:
# .p2d output summarized in a scipp DataArray
reporter = RichReporter()
intensity_p2d = powder_p2d_workflow.compute(IntensityP2D, reporter=reporter)
intensity_p2d

Below, we execute the full workflow and export the results to a .p2d file

In [ ]:
# Export the .p2d output to a file
reporter = RichReporter()
df_p2d = powder_p2d_workflow.compute(DataFrameP2D, reporter=reporter)
df_p2d